# Jev checkpoint decision benchmark

Independent record of paired interventions, evidence ablations, and blinded review. Reuses the existing SD1.5 kernel without rerunning prior experiments. All outcomes are retained.

In [ ]:
# 1. Freeze the benchmark protocol before generating candidates.
import json, time, hashlib, copy, random, math, os, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
assert 'pipe' in globals() and 'V_ROOT' in globals(), 'Select the existing hands notebook kernel.'
B_OUT=OUT/('checkpoint_benchmark_'+time.strftime('%Y%m%d_%H%M%S'));B_OUT.mkdir()
B_CONFIG={'model':'jev-1.13.0','schedule_steps':40,'checkpoint_steps':[8,20,32],
 'development_seeds':[123,271,409,557],'heldout_seeds':[701,809,919,1031,1153,1277,1409,1559],
 'intervention_steps':4,'bound_bias':.6,'temperature_options':[.85,1.,1.15],'gate_options':[.85,1.,1.15],
 'baseline':'ordinary continuation','other_baseline':'seeded random joint intervention with identical controls and duration',
 'reviewer':'Qwen/Qwen2-VL-7B-Instruct','review_status':'uncalibrated; sanity tests required; no human-verified anatomy labels',
 'variants':['correct_history','no_history','shuffled_history','renamed','reversed'],
 'endpoint_rule':'Report terminal continuation of every branch; never select a nicer intermediate frame.',
 'success_rule':'Provisional evidence only if held-out blind bidirectional review favors Jev over both comparators, reviewer sanity passes, and correct history improves over shuffled/missing history. No anatomical proof from numeric pixel change.',
 'limits':'Checkpoint cases share trajectories. Only 12 source seeds; no independent-human quality verdict. Finite bounded action menu, not freeform tensor gradients.'}
B_PROMPTS=[PROMPT,
 'Studio photograph of a single human hand, back of hand facing camera, fingers relaxed and separated, wearing a simple silver ring on the ring finger, plain dark gray background, realistic skin, sharp focus',
 'Studio photograph of a single human hand in a three-quarter view, gently curved fingers clearly visible, plain dark gray background, realistic skin, sharp focus']
def b_dump(name,value):
 text=json.dumps(value,indent=2,default=lambda x:x.item() if hasattr(x,'item') else str(x));assert _api_key not in text
 (B_OUT/name).write_text(text)
b_dump('protocol.json',{'config':B_CONFIG,'prompts':B_PROMPTS,'source_run':str(V_OUT)})
print('Artifact directory:',B_OUT)
display(pd.DataFrame({'split':['development','held-out'],'seeds':[4,8],'checkpoints':[12,24]}))
print('Seed123 original image is retained separately. Fresh noise uses the same seed123, but this 40-step sampler is a new trajectory, not a reproduction of the earlier image.')
print('Existing GPU:',torch.cuda.get_device_name(),'; transformers:',transformers.__version__)


In [ ]:
# 2. Load an independent larger reviewer and freeze verbatim prompts.
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
B_POOL=ThreadPoolExecutor(max_workers=6)
def b_load_reviewer():
 p=AutoProcessor.from_pretrained(B_CONFIG['reviewer'],min_pixels=256*28*28,max_pixels=768*28*28)
 m=Qwen2VLForConditionalGeneration.from_pretrained(B_CONFIG['reviewer'],torch_dtype=torch.float16,attn_implementation='sdpa',device_map='cpu').eval()
 return p,m
B_REVIEW_FUTURE=B_POOL.submit(b_load_reviewer)
print('Reviewer loading on CPU in background.')
B_REVIEW_PROMPT="""Compare photographs A and B against this requested scene: {prompt}
Assess physically plausible hand structure: connections to the palm, digit proportions, joints, separations and occlusion. Five visible digits alone is not sufficient, and pose can hide digits. Also assess requested jewelry/detail, photographic coherence, and unrelated damage. Do not assume the panels differ. Do not prefer sharper texture when structure is worse. Do not infer an editing method or read panel order as chronology.
Return only JSON with keys: winner, anatomy, detail, preservation (each A, B, tie, or uncertain), reason (at most 45 words naming visible evidence). Choose uncertain when differences cannot be reliably judged. This is a relative visual assessment, not a calibrated anatomy measurement."""
B_SELECT_PROMPT="""Select one executable control vector for this diffusion checkpoint. The goal is a physically plausible hand matching the scene while preserving coherent image content. Compare all candidates using their measured short-probe effects, explicit architecture metadata and available prior completed interventions. Attention entropy, feature dispersion, output norm and pixel change are diagnostics, not quality rewards; neither increasing nor decreasing them is inherently desirable. Numerical changes cannot establish anatomical repair. Visual observations are fallible and may conflict. A prior example only transfers to the extent its conditions are comparable. Candidate identifiers and listing order have no semantic significance. Pick ordinary continuation when no intervention is supported. Joint controls are applied together for four denoising steps, followed by ordinary completion. Do not assume a component corresponds to a finger. Outcomes after the current short probe are not yet supplied."""
B_PREDICT_PROMPT="""Before seeing completed outcomes, predict whether at least one offered non-neutral intervention will improve the completed image over ordinary continuation without a larger unrelated regression. Use only current measurements and explicitly available history. This forecasts the existence of a useful intervention, not the quality of any particular candidate; the questions in this request are independent."""
b_dump('verbatim_prompts.json',{'selection':B_SELECT_PROMPT,'prediction':B_PREDICT_PROMPT,'review':B_REVIEW_PROMPT})
print('SELECTION:\n'+B_SELECT_PROMPT+'\nPREDICTION:\n'+B_PREDICT_PROMPT+'\nREVIEW:\n'+B_REVIEW_PROMPT)


In [ ]:
# 3. Isolated DDIM executor and instrumented attention; no old loop is rerun.
from diffusers import DDIMScheduler
B_SCHED=DDIMScheduler.from_config(A_SCHED.config);B_SCHED.set_timesteps(40,device=pipe.device)
B_ACTIVE=None;B_CAPTURE=False;B_UNET_CALLS=0
_cls=next(n for n in ast.parse(_src_nb.cells[61].source).body if isinstance(n,ast.ClassDef))
_bsource=ast.get_source_segment(_src_nb.cells[61].source,_cls).replace('AdaptiveAttention','BenchmarkAttention').replace('V_ACTIVE','B_ACTIVE').replace('V_CAPTURE','B_CAPTURE')
exec(_bsource);(B_OUT/'attention_implementation.py').write_text(_bsource)
def b_install():
 pipe.unet.set_attn_processor({name:BenchmarkAttention(name,V_TARGETS.index(name)) if name in V_TARGETS else proc for name,proc in V_BASE_PROCESSORS.items()})
@torch.inference_mode()
def b_embed(prompt):
 toks=pipe.tokenizer([NEGATIVE,prompt],padding='max_length',max_length=77,truncation=True,return_tensors='pt').to(pipe.device)
 return pipe.text_encoder(toks.input_ids)[0]
@torch.inference_mode()
def b_predict(s,capture=False):
 global B_ACTIVE,B_CAPTURE,B_UNET_CALLS
 assert 0<=s['i']<40
 B_ACTIVE=s;B_CAPTURE=capture;B_UNET_CALLS+=1
 pred=pipe.unet(s['z'].repeat(2,1,1,1),B_SCHED.timesteps[s['i']],encoder_hidden_states=s['embedding']).sample
 u,c=pred.chunk(2);return u+CFG*(c-u)
@torch.inference_mode()
def b_steps(s,count,capture=False):
 s=v_clone(s)
 for _ in range(min(count,40-s['i'])):
  eps=b_predict(s);r=B_SCHED.step(eps,B_SCHED.timesteps[s['i']],s['z'],eta=0.)
  s['z']=r.prev_sample;s['clean']=r.pred_original_sample;s['i']+=1
 assert torch.isfinite(s['z']).all()
 s['image']=decode(s['clean'])[0];s['revision']+=1
 if capture and s['i']<40:b_predict(s,True)
 return s
@torch.inference_mode()
def b_start(seed,prompt,anchor=False):
 gen=torch.Generator(device=pipe.device).manual_seed(seed)
 noise=torch.randn((1,4,64,64),generator=gen,device=pipe.device,dtype=pipe.unet.dtype)
 clean=a_encode(RESULTS[(123,'jev')]['image']) if anchor else torch.zeros_like(noise)
 index=8 if anchor else 0
 z=B_SCHED.add_noise(clean,noise,B_SCHED.timesteps[index:index+1]) if anchor else noise*B_SCHED.init_noise_sigma
 return {'z':z,'clean':clean,'image':RESULTS[(123,'jev')]['image'].copy() if anchor else Image.new('RGB',(512,512)),
 'embedding':b_embed(prompt),'prompt':prompt,'i':index,'groups':{},'captures':{},'params':{},'revision':0,'task':'benchmark'}
def b_cpu(obj):
 if isinstance(obj,torch.Tensor):return obj.detach().cpu()
 if isinstance(obj,dict):return {k:b_cpu(v) for k,v in obj.items()}
 return obj
# Preserve original seed123 image as the starting anchor, not a newly selected sample.
B_CONFIG['seed123_source']='original RESULTS[(123,jev)] image, encoded and re-noised at index8 of the fixed40-step schedule; other seeds start from noise'
B_CONFIG['candidate_menu']=['ordinary','jev_proposal','opposite_proposal','seeded_random_joint']
B_CONFIG['history_ablation_scope']='selector only; candidate proposal shared; frozen development-only history for heldout'
b_dump('protocol.json',{'config':B_CONFIG,'prompts':B_PROMPTS,'source_run':str(V_OUT)})
RESULTS[(123,'jev')]['image'].save(B_OUT/'original_seed123_anchor.png')
_bt=b_start(123,B_PROMPTS[0],True)
pipe.unet.set_attn_processor(dict(V_BASE_PROCESSORS))
with torch.inference_mode():
 _raw=pipe.unet(_bt['z'].repeat(2,1,1,1),B_SCHED.timesteps[_bt['i']],encoder_hidden_states=_bt['embedding']).sample
 _u,_c=_raw.chunk(2);_ref=_u+CFG*(_c-_u)
 b_install();_observed=b_predict(_bt,True)
B_IDENTITY=float((_ref-_observed).abs().max());assert B_IDENTITY<.02, B_IDENTITY
b_dump('executor_preflight.json',{'neutral_prediction_max_abs':B_IDENTITY,'scheduler':dict(B_SCHED.config),'cfg':CFG})
print('Neutral wrapper max error:',B_IDENTITY,'; checkpoint feature grids:',[c['shape'] for c in _bt['captures'].values()])
print('Reviewer download ready:',B_REVIEW_FUTURE.done())


In [ ]:
# 5. Audited Jev API adapter, numerical context, and joint architectural proposals.
B_API_LOCK=threading.Lock();B_API_LOG=[]
def b_call(tag,state,questions):
 payload={'model':B_CONFIG['model'],'state':state,'questions':questions}
 b_dump(tag+'_request.json',payload);start=time.time()
 for attempt in range(4):
  r=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json=payload,timeout=90)
  if r.status_code==200:break
  if r.status_code not in (429,500,502,503,529):raise RuntimeError('Jev HTTP '+str(r.status_code)+' at '+tag)
  time.sleep(2**attempt)
 if r.status_code!=200:raise RuntimeError('Jev retry limit at '+tag)
 result=r.json();b_dump(tag+'_response.json',result);answers=result['answers']
 for k,q in questions.items():
  if q['type']=='choice':assert answers[k]['choice'] in q['criteria']
 with B_API_LOCK:B_API_LOG.append({'tag':tag,'seconds':time.time()-start,'usage':result.get('usage'),'model':result.get('model')})
 return answers
B_VECTORS={'hold':(0,1,1),'route_plus':(.4,.85,1.15),'route_minus':(-.4,1.15,.85),
 'concentrate':(0,.85,1),'broaden':(0,1.15,1),'amplify':(0,1,1.15),'suppress':(0,1,.85)}
B_PROPOSAL_PROMPT="""Propose a bounded, testable joint change to four self-attention modules for the given hand scene. For each module choose one head, a source feature group, a destination feature group, and one available control vector. All modules' choices execute simultaneously. The source/destination groups are measured similarity clusters, not anatomical labels. Read explicit group footprints, head measurements and prior measured outcomes; no entropy or norm target is intrinsically desirable. Keep a module unchanged when evidence is insufficient. The resulting vector is a hypothesis to test, not a verified repair. These questions are independent: later fields cannot see answers to earlier fields in this call. Source/destination/head choices should each be justified by the supplied state, and their combination will be tested as a whole."""
def b_context(case):
 s=case['state'];layers={}
 for l,name in enumerate(V_TARGETS):
  cap=s['captures'][name];cat=v_catalogue(s,l)
  groups={str(r['record']['parent_group']):r['record'] for r in cat.values() if r['record']['kind']=='feature_group'}
  layers[str(l)]={'module':name,'grid':cap['shape'],'groups':groups,
    'heads':{str(h):{'entropy':round(cap['entropy'][h],5),'rms':round(cap['head_rms'][h],5),'group_to_group_mass':np.round(cap['group_mass'][h],4).tolist()} for h in range(8)}}
 return {'scene':case['prompt'],'noise_index':case['index'],'schedule_length':40,'layers':layers,
 'definitions':{'group_to_group_mass':'Mean attention probability mass between groups from up to128 sampled queries; not a semantic or quality score.',
 'occupancy4x4':'Fraction of feature group membership per spatial bin; approximate footprint, not a finger label.',
 'entropy':'Natural-log self-attention entropy; RMS is head-output magnitude; values are layer-specific.'}}
def b_proposal(case,history):
 state=b_context(case);state['prior_completed_interventions']=history
 qs={}
 for l in range(4):
  for field,criteria in [('head',{str(h):f'head {h}' for h in range(8)}),('source',{str(g):f'source feature group {g}' for g in range(6)}),('destination',{str(g):f'destination feature group {g}' for g in range(6)}),('vector',{k:f'bias={v[0]}, temperature={v[1]}, head_output_gate={v[2]}' for k,v in B_VECTORS.items()})]:
   qs[f'{l}_{field}']=v_choice(B_PROPOSAL_PROMPT+f' Here choose {field} for module layers[{l}].',criteria)
 ans=b_call(case['id']+'_proposal',state,qs);spec=[]
 for l in range(4):
  spec.append({'layer':l,'head':int(ans[f'{l}_head']['choice']),'source':int(ans[f'{l}_source']['choice']),
    'destination':int(ans[f'{l}_destination']['choice']),'vector':list(B_VECTORS[ans[f'{l}_vector']['choice']])})
 return spec,state

def b_apply(s,spec):
 s=v_clone(s);s['params']={}
 for e in spec:
  name=V_TARGETS[e['layer']];shape=s['captures'][name]['shape'];bias,temp,gate=e['vector']
  assert -.6<=bias<=.6 and .8<=temp<=1.2 and .8<=gate<=1.2
  if bias==0 and temp==1 and gate==1:continue
  s['params'].setdefault(name,{})[str(e['head'])]={'bias':bias,'temperature':temp,'gate':gate,
    'source':(s['groups'][name]==e['source']).reshape(shape),'destination':(s['groups'][name]==e['destination']).reshape(shape)}
 return s

def b_other_specs(case,proposal):
 inverse=copy.deepcopy(proposal)
 for e in inverse:
  b,t,g=e['vector'];e['vector']=[-b,round(2-t,5),round(2-g,5)]
 rng=random.Random(20260922+case['seed']*100+case['index'])
 rand=[{'layer':l,'head':rng.randrange(8),'source':rng.randrange(6),'destination':rng.randrange(6),
        'vector':list(rng.choice(list(B_VECTORS.values())))} for l in range(4)]
 return {'ordinary':[],'proposal':proposal,'opposite':inverse,'random':rand}
b_dump('proposal_prompt.json',{'prompt':B_PROPOSAL_PROMPT,'vectors':B_VECTORS})
print('VERBATIM PROPOSAL PROMPT:\n'+B_PROPOSAL_PROMPT)
print('Neutral, proposal, opposite, random; all candidate endpoint images are retained regardless of selection.')


In [ ]:
# 6. Independent terminal reviewer and blind sanity checks.
import re
B_REVIEW_PROCESSOR,B_REVIEW_MODEL=B_REVIEW_FUTURE.result()
B_REVIEW_PROCESSOR.tokenizer.padding_side='left'
B_REVIEW_MODEL=B_REVIEW_MODEL.to(pipe.device)
B_REVIEW_LOG=[]
def b_parse_review(text):
 try:
  obj=json.loads(re.search(r'\{.*\}',text,re.S).group(0))
  for k in ['winner','anatomy','detail','preservation']:assert obj[k] in ['A','B','tie','uncertain']
  return obj
 except Exception:return {'winner':'uncertain','anatomy':'uncertain','detail':'uncertain','preservation':'uncertain','reason':'Unparseable reviewer output','parse_error':True}
@torch.inference_mode()
def b_reviews(jobs,large=True):
 result=[];proc=B_REVIEW_PROCESSOR if large else vlm_processor;model=B_REVIEW_MODEL if large else vlm
 for offset in range(0,len(jobs),3):
  batch=jobs[offset:offset+3];images=[];texts=[]
  for job in batch:
   pic=frame_strip([job['A'],job['B']],['A','B'],width=384);images.append(pic);pic.save(B_OUT/(job['tag']+'_pair.png'))
   messages=[{'role':'user','content':[{'type':'image'},{'type':'text','text':B_REVIEW_PROMPT.format(prompt=job['prompt'])}]}]
   texts.append(proc.apply_chat_template(messages,tokenize=False,add_generation_prompt=True))
  inputs=proc(text=texts,images=images,padding=True,return_tensors='pt').to(pipe.device)
  generated=model.generate(**inputs,max_new_tokens=170,do_sample=False)
  outputs=proc.batch_decode(generated[:,inputs.input_ids.shape[1]:],skip_special_tokens=True,clean_up_tokenization_spaces=False)
  for job,text in zip(batch,outputs):
   rec={'tag':job['tag'],'model':B_CONFIG['reviewer'] if large else VLM_ID,'raw':text,'assessment':b_parse_review(text)}
   b_dump(job['tag']+'_review.json',rec);B_REVIEW_LOG.append(rec);result.append(rec)
 return result
# No exact-pixel shortcut is used here: reviewer sees ordinary blind A/B panels.
B_SANITY_JOBS=[]
for j,seed in enumerate(B_CONFIG['development_seeds']):
 im=B_BASELINES[seed];B_SANITY_JOBS.append({'A':im,'B':im.copy(),'prompt':B_PROMPTS[j%3],'tag':f'sanity_identical_{seed}','expected':'tie'})
for j,seed in enumerate(B_CONFIG['development_seeds'][:2]):
 im=B_BASELINES[seed];blur=im.filter(ImageFilter.GaussianBlur(18))
 for reverse in [False,True]:
  B_SANITY_JOBS.append({'A':blur if reverse else im,'B':im if reverse else blur,'prompt':B_PROMPTS[j%3],
   'tag':f'sanity_blur_{seed}_{int(reverse)}','expected':'B' if reverse else 'A'})
B_SANITY_REVIEWS=b_reviews(B_SANITY_JOBS)
B_SANITY=[{'tag':j['tag'],'expected':j['expected'],'actual':r['assessment']['winner'],'pass':j['expected']==r['assessment']['winner']} for j,r in zip(B_SANITY_JOBS,B_SANITY_REVIEWS)]
B_REVIEW_PASSED=all(x['pass'] for x in B_SANITY)
b_dump('reviewer_sanity.json',{'passed':B_REVIEW_PASSED,'cases':B_SANITY,'limits':'Identity and obvious blur checks do not establish subtle anatomical judgment.'})
display(pd.DataFrame(B_SANITY));print('Reviewer sanity passed:',B_REVIEW_PASSED)
print('36 checkpoints available:',len(B_CASES),'GPU GiB allocated:',round(torch.cuda.memory_allocated()/2**30,2))


In [ ]:
# 7. Validate JSON parsing against saved raw reviews; no reviewer calls are repeated.
def b_parse_review(text):
 try:
  obj=json.loads(text[text.index('{'):text.rindex('}')+1])
  for k in ['winner','anatomy','detail','preservation']:assert obj[k] in ['A','B','tie','uncertain']
  return obj
 except Exception:return {'winner':'uncertain','anatomy':'uncertain','detail':'uncertain','preservation':'uncertain','reason':'Unparseable reviewer output','parse_error':True}
for r in B_SANITY_REVIEWS:r['assessment']=b_parse_review(r['raw']);b_dump(r['tag']+'_review_parsed.json',r)
B_SANITY=[{'tag':j['tag'],'expected':j['expected'],'actual':r['assessment']['winner'],'pass':j['expected']==r['assessment']['winner']} for j,r in zip(B_SANITY_JOBS,B_SANITY_REVIEWS)]
B_REVIEW_PASSED=all(x['pass'] for x in B_SANITY)
b_dump('reviewer_sanity.json',{'passed':B_REVIEW_PASSED,'cases':B_SANITY,'parser_note':'Reparsed original outputs by JSON brace span; no sampling or output selection.'})
display(pd.DataFrame(B_SANITY))
for r in B_SANITY_REVIEWS:print(r['tag'],r['raw'])
print('Sanity passed:',B_REVIEW_PASSED)


In [ ]:
# 8. Evidence packets, history ablations, and selection before completed outcomes.
B_CONFIG['reviewer_sanity_passed']=B_REVIEW_PASSED
B_CONFIG['variants']+=['raw_only']
B_CONFIG['quality_claim_gate']='FAILED reviewer identity sanity; all automatic quality outcomes remain provisional regardless of win rate.'
B_CONFIG['fair_selection_baseline']='Uniform random selection from the same four evaluated candidates; same probe information and diffusion candidate cost. API cost reported separately.'
b_dump('protocol.json',{'config':B_CONFIG,'prompts':B_PROMPTS,'source_run':str(V_OUT)})
def b_metrics(ref,candidate):
 out={'pixel_change':v_delta(ref['image'],candidate['image']),'layers':{}}
 for l,name in enumerate(V_TARGETS):
  a=ref['captures'][name];b=candidate['captures'][name]
  out['layers'][str(l)]={'entropy_reference':a['entropy'],'entropy_candidate':b['entropy'],
   'rms_reference':a['head_rms'],'rms_candidate':b['head_rms'],
   'derived_entropy_delta':(np.array(b['entropy'])-a['entropy']).round(6).tolist(),
   'derived_rms_delta':(np.array(b['head_rms'])-a['head_rms']).round(6).tolist()}
 return out

def b_history_for(case):
 if case['split']=='heldout':return copy.deepcopy(B_FROZEN_HISTORY)
 return copy.deepcopy(B_HISTORY[-6:])

def b_select_variant(case,context,cards,history,variant):
 names=['ordinary','proposal','opposite','random']
 ids=['n_318','n_742','n_195','n_863'] if variant!='renamed' else ['v_907','v_214','v_658','v_431']
 mapping=dict(zip(names,ids));inverse={v:k for k,v in mapping.items()}
 records={mapping[k]:copy.deepcopy(cards[k]) for k in names}
 h=copy.deepcopy(history)
 if variant=='no_history':h=[]
 if variant=='shuffled_history':
  # Permute completed outcome labels among non-neutral interventions, retaining real measurements and controls.
  for entry in h:
   outs=entry.get('outcomes',{});keys=[k for k in ['proposal','opposite','random'] if k in outs]
   vals=[copy.deepcopy(outs[k]) for k in keys]
   for i,k in enumerate(keys):outs[k]=vals[(i+1)%len(vals)]
 if variant=='raw_only':
  for card in records.values():
   for l in card['short_probe']['layers'].values():l.pop('derived_entropy_delta',None);l.pop('derived_rms_delta',None)
 if variant=='reversed':records=dict(reversed(list(records.items())))
 state={'scene':case['prompt'],'noise_index':case['index'],'schedule_steps':40,'architecture':context['layers'],
 'definitions':context['definitions'],'candidates':records,'prior_completed_interventions':h,
 'reviewer_limit':'Short-probe observer is Qwen2-VL-2B, which previously failed identity sanity. Treat its text as uncertain. Numerical differences are not anatomy scores.',
 'execution':'All candidate controls operate for four steps, then reset to neutral for ordinary terminal completion. All start from identical latent and use deterministic eta=0.'}
 qs={'selection':v_choice(B_SELECT_PROMPT,{k:f'Execute the control vector in candidates[{k}].' for k in records}),
     'useful_intervention':{'type':'noul','instructions':B_PREDICT_PROMPT}}
 ans=b_call(case['id']+'_select_'+variant,state,qs)
 return {'variant':variant,'choice':inverse[ans['selection']['choice']],
  'probabilities':{inverse[k]:v for k,v in ans['selection']['probabilities'].items()},
  'forecast':ans['useful_intervention'],'history_entries':len(h),
  'history_hash':hashlib.sha256(json.dumps(h,sort_keys=True).encode()).hexdigest()}

def b_bidirectional_verdict(ab,ba):
 a=ab['assessment'];b=ba['assessment']
 if a.get('parse_error') or b.get('parse_error'):return 'unresolved'
 if a['winner']=='B' and b['winner']=='A' and a['anatomy']!='A' and b['anatomy']!='B':return 'win'
 if a['winner']=='A' and b['winner']=='B':return 'loss'
 if a['winner']=='tie' and b['winner']=='tie':return 'tie'
 return 'unresolved'
print('Six selection variants run concurrently; candidate identity is mapped back before comparisons.')
print('Review gate failed: automatic verdicts will be diagnostic only. Both panel orders required for directional verdicts.')


In [ ]:
# 9. Run every case; record choices before terminal images are generated/reviewed.
B_FROZEN_HISTORY=[];B_PROGRESS={'completed':0,'total':36,'phase':'ready'}
def b_progress(case,phase):
 B_PROGRESS.update(case=case['id'],phase=phase,completed=len(B_RECORDS),unet_calls=B_UNET_CALLS,api_calls=len(B_API_LOG))
 b_dump('live_status.json',B_PROGRESS)
@torch.inference_mode()
def b_run_case(case):
 started=time.time();history=b_history_for(case);b_progress(case,'joint proposal')
 proposal,context=b_proposal(case,history);specs=b_other_specs(case,proposal)
 early={};short={};cards={}
 b_progress(case,'matched short probes')
 for name,spec in specs.items():
  a=b_steps(b_apply(case['state'],spec),2);b=b_steps(a,2,capture=True)
  early[name]=a;short[name]=b
  a['image'].save(B_OUT/(case['id']+'_'+name+'_step2.png'));b['image'].save(B_OUT/(case['id']+'_'+name+'_step4.png'))
 shortjobs=[{'A':short['ordinary']['image'],'B':short[name]['image'],'prompt':case['prompt'],'tag':case['id']+'_short_'+name} for name in ['proposal','opposite','random']]
 b_progress(case,'short-probe observer')
 observed=b_reviews(shortjobs,large=False)
 for name in specs:
  report={'winner':'tie','reason':'Reference branch'} if name=='ordinary' else observed[['proposal','opposite','random'].index(name)]['assessment']
  cards[name]={'controls':specs[name],'short_probe':b_metrics(short['ordinary'],short[name]),'visual_estimate':report,
   'visual_panel_mapping':'A is ordinary continuation; B is this candidate. Fallible short-probe observation only.'}
 b_dump(case['id']+'_candidate_evidence.json',{'context':context,'cards':cards,'history':history})
 b_progress(case,'parallel selections; terminal outcomes unseen')
 futures={variant:B_POOL.submit(b_select_variant,case,context,cards,history,variant) for variant in B_CONFIG['variants']}
 selections={variant:f.result() for variant,f in futures.items()}
 b_dump(case['id']+'_locked_selections.json',{'selections':selections,'terminal_outcomes_generated':False,'timestamp':time.time()})
 b_progress(case,'terminal completion')
 final={}
 for name in specs:
  s=short[name];s['params']={};end=b_steps(s,40-s['i']);final[name]=end['image']
  end['image'].save(B_OUT/(case['id']+'_'+name+'_final.png'))
  torch.save(end['z'].detach().cpu(),B_OUT/(case['id']+'_'+name+'_final_latent.pt'))
 assert np.array_equal(np.asarray(final['ordinary']),np.asarray(B_BASELINES[case['seed']])), 'Ordinary replay mismatch'
 b_progress(case,'blind terminal reviews in both panel orders')
 jobs=[]
 for name in ['proposal','opposite','random']:
  jobs.extend([{'A':final['ordinary'],'B':final[name],'prompt':case['prompt'],'tag':case['id']+'_terminal_'+name+'_AB'},
               {'A':final[name],'B':final['ordinary'],'prompt':case['prompt'],'tag':case['id']+'_terminal_'+name+'_BA'}])
 reviews=b_reviews(jobs,large=True);verdicts={'ordinary':'tie'};terminal={}
 for j,name in enumerate(['proposal','opposite','random']):
  exact=np.array_equal(np.asarray(final[name]),np.asarray(final['ordinary']))
  verdicts[name]='tie' if exact else b_bidirectional_verdict(reviews[2*j],reviews[2*j+1])
  terminal[name]={'verdict':verdicts[name],'exact_pixel_identity':exact,'AB':reviews[2*j]['assessment'],'BA':reviews[2*j+1]['assessment'],
   'pixel_change':v_delta(final['ordinary'],final[name])}
 rec={'id':case['id'],'split':case['split'],'seed':case['seed'],'index':case['index'],'prompt':case['prompt'],
 'specs':specs,'selections':selections,'verdicts':verdicts,'terminal_reviews':terminal,'seconds':time.time()-started,
 'reviewer_validated':B_REVIEW_PASSED,'quality_claim':'unvalidated automated review only'}
 b_dump(case['id']+'_result.json',rec)
 if case['split']=='development':
  B_HISTORY.append({'scene':case['prompt'],'noise_index':case['index'],'controls':specs,'outcomes':{k:terminal[k] for k in terminal},
   'reviewer_warning':'Reviewer failed identical-image sanity. These are provisional relative judgments, not ground truth.'})
 grid=frame_strip([final[n] for n in specs],[n for n in specs],width=256);grid.save(B_OUT/(case['id']+'_all_endpoints.png'))
 return rec

def b_run_all():
 global B_FROZEN_HISTORY
 for case in B_CASES:
  if any(r['id']==case['id'] for r in B_RECORDS):continue
  if case['split']=='heldout' and not B_FROZEN_HISTORY:
   # Freeze one history example at each noise level per development seed; all12 retained.
   B_FROZEN_HISTORY=copy.deepcopy(B_HISTORY);b_dump('frozen_development_history.json',B_FROZEN_HISTORY)
  rec=b_run_case(case);B_RECORDS.append(rec)
  b_dump('all_results.json',B_RECORDS);b_dump('api_usage.json',B_API_LOG)
  b_progress(case,'case complete')
  pick=rec['selections']['correct_history']['choice']
  print(f"{len(B_RECORDS):02}/36 {case['id']} | selected={pick} | provisional={rec['verdicts'][pick]} | {rec['seconds']:.1f}s",flush=True)
 B_PROGRESS.update(phase='finished',completed=len(B_RECORDS));b_dump('live_status.json',B_PROGRESS)
 print('DONE. All cases, choices and images retained at',B_OUT.resolve())
# Archive every executed benchmark source, including the checkpoint-generation cell from kernel history.
for n,src in enumerate(In):
 if any(src.startswith('# '+str(k)+'. ') for k in range(1,10)) and ('B_' in src):
  (B_OUT/f'executed_source_{n:03}.py').write_text(src)
print('Frozen run ready. Development12 + heldout24. No prompt tuning after results are inspected.')


In [ ]:
# 10. Launch the full benchmark (paid Jev calls and GPU inference).
# Retrieval rule fixed before any candidate outcomes: four closest available development records.
def b_history_for(case):
 pool=B_FROZEN_HISTORY if case['split']=='heldout' else B_HISTORY
 ranked=sorted(enumerate(pool),key=lambda iv:(iv[1]['noise_index']!=case['index'],iv[1]['scene']!=case['prompt'],-iv[0]))
 return copy.deepcopy([x for _,x in ranked[:4]])
B_CONFIG['history_retrieval']='Up to4 prior development records, exact noise index first, then exact prompt, then most recent. Heldout pool frozen after12 development cases.'
b_dump('protocol.json',{'config':B_CONFIG,'prompts':B_PROMPTS,'source_run':str(V_OUT)})
# Prevent incidental generation-config warnings; decoding remains greedy.
for model in [vlm,B_REVIEW_MODEL]:
 model.generation_config.temperature=1.;model.generation_config.top_p=1.;model.generation_config.top_k=50
vlm_processor.tokenizer.padding_side='left'
print('START',time.strftime('%H:%M:%S'),'| all36 cases | 6 concurrent selection variants per case',flush=True)
b_run_all()


In [ ]:
# 11. Diagnose the failed request; retain development results and error evidence.
_failed=json.loads((B_OUT/'development_271_20_select_correct_history_request.json').read_text())
_err=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json=_failed,timeout=90)
print('HTTP status:',_err.status_code)
print(_err.text[:3000] if _api_key not in _err.text else 'Credential redacted')
b_dump('development_transport_failure.json',{'status':_err.status_code,'body':_err.text,'completed_case_ids':[r['id'] for r in B_RECORDS]})
print('Payload characters:',len(json.dumps(_failed)),'State characters:',len(json.dumps(_failed['state'])))
print('Retained cases:',len(B_RECORDS))


In [ ]:
# 12. Development-only engineering repair: smaller evidence packets; preserve original attempts.
import shutil
_attempt=B_OUT/'development_attempt_before_compaction';_attempt.mkdir(exist_ok=True)
for p in list(B_OUT.glob('development_271_20*')):
 if p.is_file():shutil.copy2(p,_attempt/p.name)
for r in B_RECORDS:r['context_version']='full_v1'
b_dump('completed_before_compaction.json',B_RECORDS)
_b_call_uncompacted=b_call

def b_round_context(x):
 if isinstance(x,float):return round(x,4)
 if isinstance(x,dict):return {k:b_round_context(v) for k,v in x.items()}
 if isinstance(x,list):return [b_round_context(v) for v in x]
 return x

def b_compact_context(state):
 s=copy.deepcopy(state)
 if 'prior_completed_interventions' in s:
  for entry in s['prior_completed_interventions']:
   for name,out in list(entry.get('outcomes',{}).items()):
    entry['outcomes'][name]={k:out[k] for k in ['verdict','exact_pixel_identity','pixel_change'] if k in out}
    for order in ['AB','BA']:
     if order in out:
      a=copy.deepcopy(out[order]);reason=a.get('reason','');a['reason']=reason[:240];a['reason_truncated']=len(reason)>240
      entry['outcomes'][name][order]=a
 if 'candidates' in s:
  selected={str(l):{'heads':set(),'groups':set()} for l in range(4)}
  for candidate in s['candidates'].values():
   compact_layers={}
   for e in candidate['controls']:
    l=str(e['layer']);h=e['head'];raw=candidate['short_probe']['layers'][l]
    compact_layers[l]={'head':h,**{k:values[h] for k,values in raw.items()}}
    selected[l]['heads'].add(str(h));selected[l]['groups'].update([str(e['source']),str(e['destination'])])
   candidate['short_probe']['layers']=compact_layers
  for l,layer in s['architecture'].items():
   layer['heads']={h:v for h,v in layer['heads'].items() if h in selected[l]['heads']}
   layer['groups']={g:v for g,v in layer['groups'].items() if g in selected[l]['groups']}
 s['encoding_note']='Displayed floating measurements rounded to4 decimals; complete raw tensors and reports retained in artifacts. Selector includes only heads/groups used by offered controls. Long prior review reasons capped at240 characters with truncation flag.'
 return b_round_context(s)

def b_call(tag,state,questions):
 compact=b_compact_context(state)
 return _b_call_uncompacted(tag+'_compact_v2',compact,questions)
B_CONFIG['context_version']='compact_v2'
B_CONFIG['development_amendment']='API max_tokens_exceeded on fifth development case; retain first4 v1 cases and failed-attempt artifacts. All heldout cases use fixed compact_v2. No action menu or quality rubric changed.'
b_dump('protocol.json',{'config':B_CONFIG,'prompts':B_PROMPTS,'source_run':str(V_OUT)})
print('Rejected state chars:',len(json.dumps(_failed['state'])), 'Compact state chars:',len(json.dumps(b_compact_context(_failed['state']))))
print('Resume from case',len(B_RECORDS)+1,'; heldout has not started.')
b_run_all()


In [ ]:
# 13. Analyze all completed cases; this cell runs after the benchmark finishes.
assert len(B_RECORDS)==36, 'Do not report a partial run as complete.'
import collections, html
from IPython.display import HTML
B_HASHES={}
for r in B_RECORDS:
 for name in r['specs']:
  im=Image.open(B_OUT/(r['id']+'_'+name+'_final.png')).convert('RGB')
  B_HASHES[(r['id'],name)]=hashlib.sha256(im.tobytes()).hexdigest()
B_COUNTS=[];B_STABILITY=[];B_CHOICES=[]
for split in ['development','heldout']:
 subset=[r for r in B_RECORDS if r['split']==split]
 for variant in B_CONFIG['variants']:
  outcomes=collections.Counter(r['verdicts'][r['selections'][variant]['choice']] for r in subset)
  choices=collections.Counter(r['selections'][variant]['choice'] for r in subset)
  B_COUNTS.append({'split':split,'policy':variant,'n':len(subset),**{k:outcomes[k] for k in ['win','loss','tie','unresolved']}})
  B_CHOICES.append({'split':split,'policy':variant,**{k:choices[k] for k in ['ordinary','proposal','opposite','random']}})
 for name in ['ordinary','proposal','opposite','random']:
  counts=collections.Counter(r['verdicts'][name] for r in subset)
  B_COUNTS.append({'split':split,'policy':'always_'+name,'n':len(subset),**{k:counts[k] for k in ['win','loss','tie','unresolved']}})
 uniform={k:sum(sum(r['verdicts'][a]==k for a in r['specs'])/4 for r in subset) for k in ['win','loss','tie','unresolved']}
 B_COUNTS.append({'split':split,'policy':'uniform_same_candidate_menu_EXPECTED','n':len(subset),**uniform})
 for variant in [v for v in B_CONFIG['variants'] if v!='correct_history']:
  changes=0;image_changes=0;tvs=[];history_changes=0
  for r in subset:
   a=r['selections']['correct_history'];b=r['selections'][variant]
   changes+=a['choice']!=b['choice'];image_changes+=B_HASHES[(r['id'],a['choice'])]!=B_HASHES[(r['id'],b['choice'])]
   tvs.append(.5*sum(abs(a['probabilities'].get(k,0)-b['probabilities'].get(k,0)) for k in r['specs']))
   history_changes+=a['history_hash']!=b['history_hash']
  B_STABILITY.append({'split':split,'variant':variant,'n':len(subset),'choice_changes':changes,'endpoint_changes':image_changes,
    'mean_probability_TV':float(np.mean(tvs)),'history_payload_changed':history_changes})
B_COUNTS_DF=pd.DataFrame(B_COUNTS);B_STABILITY_DF=pd.DataFrame(B_STABILITY);B_CHOICES_DF=pd.DataFrame(B_CHOICES)
for name,df in [('provisional_outcomes',B_COUNTS_DF),('decision_sensitivity',B_STABILITY_DF),('choice_counts',B_CHOICES_DF)]:df.to_csv(B_OUT/(name+'.csv'),index=False)
B_NONNEUTRAL=sum(any(e['vector']!=[0,1,1] for e in r['specs']['proposal']) for r in B_RECORDS)
B_CHANGED=sum(not r['terminal_reviews']['proposal']['exact_pixel_identity'] for r in B_RECORDS)
B_SUMMARY={'completed_cases':36,'development_cases':12,'heldout_cases':24,'source_seeds':12,'prompts':3,
 'neutral_wrapper_max_error':B_IDENTITY,'reviewer_sanity_passed':B_REVIEW_PASSED,'reviewer_sanity':B_SANITY,
 'nonneutral_proposals':B_NONNEUTRAL,'proposals_changed_terminal_pixels':B_CHANGED,
 'successful_jev_requests_including_development_retry':len(B_API_LOG),'unet_calls':B_UNET_CALLS,'raw_visual_reviews':len(B_REVIEW_LOG),
 'heldout_choice_counts':[r for r in B_CHOICES if r['split']=='heldout'],'sensitivity':B_STABILITY,
 'provisional_outcomes':B_COUNTS,'claim':'No validated image-quality gain established: reviewer failed identical-image sanity.',
 'limitations':['One shared SD1.5 model, 12 source seeds, three correlated checkpoints per seed.','No independent human anatomy labels.',
  'History ablation changes selection only, not candidate proposals.','Four-step bounded attention edits; no magnification or gradient computation in this benchmark.',
  'First4 development cases use v1 context; remaining development and all heldout cases use compact_v2 after API size rejection.',
  'History/candidate evidence rounded to4 decimal places in API context; raw tensors retained.','No self-consistency ensemble applied: variants are measured separately.']}
b_dump('summary.json',B_SUMMARY);b_dump('all_results.json',B_RECORDS);b_dump('api_usage.json',B_API_LOG)
display(Markdown('## Completed checkpoint benchmark\n\n**36/36 cases completed. Reviewer validation FAILED: automated quality verdicts below are provisional, not evidence of anatomical improvement.**'))
print('Non-neutral Jev proposals:',B_NONNEUTRAL,'/36; changed terminal pixels:',B_CHANGED,'/36')
print('HELD-OUT CHOICES');display(B_CHOICES_DF[B_CHOICES_DF.split=='heldout'])
print('DECISION SENSITIVITY');display(B_STABILITY_DF)
print('PROVISIONAL HELD-OUT OUTCOMES versus ordinary continuation; unresolved retained');display(B_COUNTS_DF[B_COUNTS_DF.split=='heldout'])
print('Forecast output example (not calibrated against unreliable reviewer):',B_RECORDS[0]['selections']['correct_history']['forecast'])
print('UNet calls:',B_UNET_CALLS,'Successful Jev calls including development retry:',len(B_API_LOG))
# Every completed endpoint remains visible. Selected branch is a record, not a hand-picked winner.
B_FILES_BASE='/files/workspace/crazy_exp/'+B_OUT.as_posix()+'/'
def b_gallery(prefix):
 parts=['<h2>All36 checkpoint cases — every terminal branch</h2><p><b>Automatic reviewer failed identity sanity.</b> No best-looking frame was substituted. Columns: ordinary, Jev proposal, opposite proposal, random joint edit.</p>']
 for r in B_RECORDS:
  selected=r['selections']['correct_history']['choice'];opened=' open' if r['seed']==123 else ''
  title=html.escape(f"{r['id']} | Jev chose {selected} | provisional verdict: {r['verdicts'][selected]}")
  parts.append(f'<details{opened}><summary>{title}</summary><p>{html.escape(r["prompt"])}</p><img loading="lazy" style="width:100%;max-width:1100px" src="{prefix}{r["id"]}_all_endpoints.png"><p>Selection variants: {html.escape(str({k:v["choice"] for k,v in r["selections"].items()}))}</p></details>')
 return '\n'.join(parts)
(B_OUT/'all_endpoints.html').write_text('<!doctype html><meta charset="utf-8"><title>Jev checkpoint benchmark</title>'+b_gallery(''))
display(HTML(b_gallery(B_FILES_BASE)))
# Export reproducibility sources without executing old experiment loops.
for n,src in enumerate(In):
 if src.startswith('# ') and ('B_' in src) and any(src.startswith('# '+str(k)+'. ') for k in range(1,14)):
  (B_OUT/f'executed_source_{n:03}.py').write_text(src)
print('All artifacts:',B_OUT.resolve())


In [ ]:
# 14. Artifact index and reproducibility record (no additional inference).
assert len(B_RECORDS)==36
B_DEPENDENCIES={'diffusion_model':dict(pipe.config),'scheduler':dict(B_SCHED.config),'torch':torch.__version__,
 'transformers':transformers.__version__,'gpu':torch.cuda.get_device_name(),'source_notebook':'Jev_Attention_Downsampling_Hands.ipynb',
 'execution':'Shares the already-loaded source notebook kernel; setup asserts pipe and V_ROOT exist. Do not run all cells expecting an independent fresh-kernel bootstrap.'}
b_dump('runtime_dependencies.json',B_DEPENDENCIES)
B_README='''# Jev checkpoint decision benchmark

36 completed cases: 12 development checkpoints and 24 held-out checkpoints; 12 source seeds, 3 poses, 3 checkpoint times per seed. Original seed123 image retained as an encoded/re-noised development anchor.

**No validated image-quality improvement established.** Qwen2-VL-7B-Instruct failed 3 of 4 blind identical-image comparisons. Bidirectional endpoint judgments are diagnostic only. Exact pixel identity overrides invented differences for scoring but the raw model answers remain preserved.

## Files
- protocol.json: fixed controls, split, success rule and recorded development-only context-size repair.
- verbatim_prompts.json and proposal_prompt.json: exact Jev/reviewer instructions.
- case_manifest.json and *_state.pt / *_internals.pt: all input checkpoints, raw tensor measurements.
- *_request.json and *_response.json: full Jev payloads and returned probabilities (no credentials).
- *_locked_selections.json: choices saved before terminal outcomes were generated.
- *_step2.png / *_step4.png / *_final.png: every candidate progression.
- *_terminal_*_review.json: blinded reviewer responses in both panel orders.
- *_result.json and all_results.json: complete per-case records.
- decision_sensitivity.csv / choice_counts.csv / provisional_outcomes.csv / summary.json: aggregate diagnostics.
- all_endpoints.html: complete image gallery; the notebook embeds the same gallery.
- reviewer_sanity.json and sanity_*: identity/blur tests, including failed judgments.
- development_attempt_before_compaction/: preserved failed fifth-case attempt before API context-size repair.
- executed_source_*.py: executed notebook sources, including checkpoint generation recovered from kernel history.

## Reproduction limits
This notebook intentionally shares the loaded source notebook kernel. Original model loading and helpers come from Jev_Attention_Downsampling_Hands.ipynb; this is not a standalone fresh-kernel notebook. Checkpoints, raw internals, exact requests and model/runtime metadata are retained for independent reanalysis without repeating paid inference.

No held-out result is fed back as intervention history. The first four development cases use the full context; all held-out cases use compact_v2. History ablations affect selection only; the proposal and candidate set remain shared. Arithmetic summaries are computed in code and do not measure anatomy. No magnification, latent optimization or computed gradients are tested here.
'''
(B_OUT/'README.md').write_text(B_README)
_sources=[]
for n,src in enumerate(In):
 if src.startswith('# ') and 'B_' in src and any(src.startswith('# '+str(k)+'. ') for k in range(1,15)):
  p=B_OUT/f'executed_source_{n:03}.py';p.write_text(src);_sources.append({'execution':n,'file':p.name,'sha256':hashlib.sha256(src.encode()).hexdigest()})
b_dump('source_manifest.json',_sources)
# Restore checkpoint-generation source to the visible record without rerunning it.
_gen=next(src for src in In if src.startswith('# 4. Materialize all checkpoint states'))
display(HTML('<details><summary>Executed checkpoint-generation source (retained from kernel history)</summary><pre>'+html.escape(_gen)+'</pre></details>'))
display(Markdown('Artifacts include every endpoint, exact prompt, probability, failed reviewer check, and the development API error. See the completed-results tables and expandable gallery immediately above.'))
print('Artifact root:',B_OUT.resolve())
print('Raw reviewer sanity:',sum(x['pass'] for x in B_SANITY[:4]),'/4 identical; ',sum(x['pass'] for x in B_SANITY[4:]),'/4 blur/order checks.')


In [ ]:
# 15. Final interpretation: distinguish consistency from useful control.
B_ALL_ORDINARY=sum(v['choice']=='ordinary' for r in B_RECORDS for v in r['selections'].values())
B_RANDOM_CHANGED=sum(not r['terminal_reviews']['random']['exact_pixel_identity'] for r in B_RECORDS)
B_RANDOM_DELTA=[r['terminal_reviews']['random']['pixel_change']['mean_abs_rgb'] for r in B_RECORDS]
assert B_ALL_ORDINARY==216 and B_NONNEUTRAL==0 and B_CHANGED==0
B_FINAL_READOUT=f'''## Result: no Jev-induced improvement in this benchmark

**All36 cases completed (12 development +24 held-out).**

- Jev selected the neutral control vector at all144 module decisions: **0/36 non-neutral proposals**.
- All six selection variants chose ordinary continuation: **{B_ALL_ORDINARY}/216 choices**.
- Jev proposal endpoints were pixel-identical to ordinary continuation: **36/36**. This directly establishes zero additional image change from Jev here, independently of reviewer reliability.
- Seeded random controls changed terminal pixels in **{B_RANDOM_CHANGED}/36** cases. The executor can change images; pixel change does not establish improvement.
- Removing/shuffling history, renaming candidates, reversing order, and removing computed differences changed no winning choices. Probabilities moved, but a constant ordinary-continuation policy does not demonstrate evidence-sensitive control.
- The separate7B reviewer invented preferences on **3/4 identical-image pairs**. Its other quality judgments cannot establish anatomical gains.

**Interpretation:** This setup did not demonstrate useful Jev control. One plausible contributor is the prompt's requirement for supporting evidence before editing, combined with weak visual feedback and no established useful interventions. The experiment does not isolate that explanation, and it does not show that Jev cannot ever control inference.

A next development test should separate selecting a bounded information-gathering experiment from approving an edit, and establish a trustworthy visual judgment channel. Forcing an edit would test execution/exploration, not prove improvement. The held-out results above must remain fixed.

Every branch and failed check is retained. The 144 endpoint files include duplicates deliberately; no final attempt was cherry-picked. This is a checkpoint/attention benchmark, not a test of magnification or gradients.
'''
b_dump('final_diagnostic.json',{'ordinary_choices':B_ALL_ORDINARY,'nonneutral_proposals':B_NONNEUTRAL,
 'random_endpoints_changed':B_RANDOM_CHANGED,'random_mean_abs_rgb':{'min':min(B_RANDOM_DELTA),'mean':float(np.mean(B_RANDOM_DELTA)),'max':max(B_RANDOM_DELTA)}})
(B_OUT/'FINAL_READOUT.md').write_text(B_FINAL_READOUT)
display(Markdown(B_FINAL_READOUT))
print('Random control mean absolute RGB difference (0-1 units):',min(B_RANDOM_DELTA),float(np.mean(B_RANDOM_DELTA)),max(B_RANDOM_DELTA))
_seed_case=next(r for r in B_RECORDS if r['id']=='development_123_08')
_seed_selected=Image.open(B_OUT/(_seed_case['id']+'_'+_seed_case['selections']['correct_history']['choice']+'_final.png'))
_final_strip=frame_strip([RESULTS[(123,'jev')]['image'],B_BASELINES[123],_seed_selected],['Original seed123 anchor','Ordinary continuation','Jev selected: identical to ordinary'],width=320)
_final_strip.save(B_OUT/'seed123_final_readout.png');display(_final_strip)
print('Completed:',len(B_RECORDS),'cases;',len(B_API_LOG),'successful Jev requests;',B_UNET_CALLS,'UNet evaluations.')
